# Ground Truth → CodeMeta Formatter

Dieses Notebook enthält die Logik, um Label-Studio-Exporte in CodeMeta-3.0-JSONs zu konvertieren.
Die Implementierung ist eingebettet, damit keine separate .py-Datei notwendig ist.

In [1]:
# Grundlegende Importe und Konfiguration
import json
import os
import logging
import re
from collections import defaultdict
from typing import Any, Dict, List

GROUND_TRUTH_DIR = '../data/evaluation_ground_truth_exports'
OUTPUT_DIR = '../pipelines/evaluation/extrated_codemeta_files'
os.makedirs(OUTPUT_DIR, exist_ok=True)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("GroundTruthFormatter")

print(f'Input directory: {GROUND_TRUTH_DIR}')
print(f'Output directory: {OUTPUT_DIR}')

Input directory: ../data/evaluation_ground_truth_exports
Output directory: ../pipelines/evaluation/extrated_codemeta_files


## Helper-Funktionen und Formatter

In [4]:
def extract_project_name(repo_field: str) -> str:
    if not repo_field:
        return 'unknown'
    m = re.match(r'^\d+_(.+)$', repo_field)
    return m.group(1) if m else repo_field

def _is_url(s: str) -> bool:
    return isinstance(s, str) and s.strip().startswith(('http://', 'https://'))

def _norm(s: str) -> str:
    return (s or '').strip()


def _uniq_preserve(items, key=lambda x: x, normalize=True):
    """Remove duplicates while preserving order.
    
    Args:
        items: List of items to deduplicate
        key: Function to extract the comparison key from each item
        normalize: If True, normalize string keys to lowercase for case-insensitive comparison
    
    Returns:
        List of unique items in original order
    """
    seen = set()
    out = []
    for it in items:
        k = key(it) if callable(key) else key
        
        if normalize and isinstance(k, str):
            k = k.lower().strip()
        
        if k not in seen:
            seen.add(k)
            out.append(it)
    
    return out

def create_codemeta_structure(project_name: str, annotations: List[Dict[str, Any]]) -> Dict[str, Any]:
    codemeta: Dict[str, Any] = {
        '@context': 'https://w3id.org/codemeta/3.0',
        '@type': 'SoftwareSourceCode',
        'name': project_name,
    }

    ordered: List[tuple[str, str, int]] = []  # (label, text, start_position)
    for ann in annotations:
        for res in ann.get('result', []):
            val = res.get('value', {})
            labels = val.get('labels', [])
            text = _norm(val.get('text', ''))
            start = val.get('start', 0)
            if not text or not labels:
                continue
            for lab in labels:
                ordered.append((lab.upper(), text, start))

    # Sort by text position to preserve document order
    ordered.sort(key=lambda x: x[2])
    
    # Remove position info for processing
    ordered = [(lab, txt) for lab, txt, _ in ordered]

    licenses = []
    software_requirements = []
    citations = []
    references = []
    builds = []
    runtimes = []
    os_list = []
    related_links = []
    continuous_integration = []
    others = defaultdict(list)

    i = 0
    while i < len(ordered):
        lab, txt = ordered[i]
        
        if lab == 'SOFTWARE_REQUIREMENTS':
            entry: Dict[str, Any] = {'@type': 'SoftwareApplication', 'name': txt}
            j = i + 1
            
            # Collect all consecutive VERSION and URL entries (they can be in any order)
            while j < len(ordered) and ordered[j][0] in ('SOFTWARE_REQUIREMENTS_VERSION', 'SOFTWARE_REQUIREMENTS_URL'):
                nl, nt = ordered[j]
                
                if nl == 'SOFTWARE_REQUIREMENTS_VERSION':
                    entry['version'] = nt
                elif nl == 'SOFTWARE_REQUIREMENTS_URL' and _is_url(nt):
                    entry['url'] = nt
                
                j += 1
            
            software_requirements.append(entry)
            i = j
            continue

        elif lab == 'SOFTWARE_REQUIREMENTS_URL':
            # This URL should have been attached to a requirement already
            # If it's standalone, try to attach it to the last requirement
            if _is_url(txt):
                if software_requirements and 'url' not in software_requirements[-1]:
                    software_requirements[-1]['url'] = txt
                else:
                    others['softwareRequirementsUrl'].append(txt)

        elif lab == 'SOFTWARE_REQUIREMENTS_VERSION':
            # This version should have been attached to a requirement already
            # If it's standalone, try to attach it to the last requirement
            if software_requirements and 'version' not in software_requirements[-1]:
                software_requirements[-1]['version'] = txt
            else:
                others['softwareRequirementsVersion'].append(txt)

        elif lab == 'LICENSE':
            if _is_url(txt):
                licenses.append(txt)
            else:
                licenses.append({'@type': 'CreativeWork', 'name': txt})

        elif lab == 'LICENSE_URL':
            licenses.append(txt if _is_url(txt) else {'@type': 'CreativeWork', 'name': txt})

        elif lab in ('CITATION', 'CITATION_URL'):
            citations.append(txt if _is_url(txt) else {'@type': 'CreativeWork', 'text': txt})

        elif lab == 'CITATION_BIBTEX':
            citations.append({'@type': 'CreativeWork', 'bibliographicCitation': txt})

        elif lab in ('REFERENCE_PUBLICATION', 'REFERENCE_PUBLICATION_URL'):
            references.append({'@type': 'ScholarlyArticle', 'url': txt} if _is_url(txt) else {'@type': 'ScholarlyArticle', 'text': txt})

        elif lab == 'REFERENCE_PUBLICATION_BIBTEX':
            references.append({'@type': 'ScholarlyArticle', 'bibliographicCitation': txt})

        elif lab in ('BUILD_INSTRUCTIONS', 'DOCUMENTATION_URL'):
            builds.append(txt)

        elif lab == 'RUNTIME_PLATFORM':
            runtimes.append(txt)

        elif lab == 'OPERATING_SYSTEM':
            os_list.append(txt)
        elif lab == 'RELATED_LINK':
            related_links.append(txt)
        elif lab == 'CONTINUOUS_INTEGRATION':
            continuous_integration.append(txt)
        else:
            others[lab.lower()].append(txt)

        i += 1

    licenses = _uniq_preserve(licenses, key=lambda x: x if isinstance(x, str) else x.get('name') or x.get('url') or str(x))
    # Smart deduplication: keep duplicates only if they have different URLs/versions
    software_requirements = _deduplicate_software_requirements(software_requirements)
    citations = _uniq_preserve(citations, key=lambda x: x if isinstance(x, str) else x.get('bibliographicCitation') or x.get('text') or x.get('url') or str(x))
    references = _uniq_preserve(references, key=lambda x: x.get('url') or x.get('bibliographicCitation') or x.get('text') or str(x))
    builds = _uniq_preserve(builds)
    runtimes = _uniq_preserve(runtimes)
    os_list = _uniq_preserve(os_list)
    related_links = _uniq_preserve(related_links)
    continuous_integration = _uniq_preserve(continuous_integration)

    if licenses:
        codemeta['license'] = licenses[0] if len(licenses) == 1 else licenses
    if software_requirements:
        codemeta['softwareRequirements'] = software_requirements if len(software_requirements) > 1 else software_requirements[0]
    if citations:
        codemeta['citation'] = citations if len(citations) > 1 else citations[0]
    if references:
        codemeta['referencePublication'] = references if len(references) > 1 else references[0]
    if builds:
        codemeta['buildInstructions'] = builds if len(builds) > 1 else builds[0]
    if runtimes:
        codemeta['runtimePlatform'] = runtimes if len(runtimes) > 1 else runtimes[0]
    if os_list:
        codemeta['operatingSystem'] = os_list if len(os_list) > 1 else os_list[0]
    if related_links:
        codemeta['relatedLink'] = related_links if len(related_links) > 1 else related_links[0]
    if continuous_integration:
        codemeta['continuousIntegration'] = continuous_integration if len(continuous_integration) > 1 else continuous_integration[0]

    for k, v in others.items():
        if v:
            codemeta[k] = v if len(v) > 1 else v[0]

    return codemeta


def _deduplicate_software_requirements(reqs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Deduplicate software requirements intelligently.
    
    Rules:
    - If same name: keep only if they have different URLs or versions
    - Remove pure duplicates (same name, url, version)
    - If one is standalone (no url/version) and another has metadata, remove standalone
    - Prefer entries with metadata over standalone
    """
    if not reqs:
        return reqs
    

    seen = {}  # name -> list of entries
    
    for req in reqs:
        name = req.get('name', '').lower().strip()
        if not name:
            continue
        
        if name not in seen:
            seen[name] = []
        seen[name].append(req)
    
    result = []
    
    for name, entries in seen.items():
        if len(entries) == 1:
            result.append(entries[0])
            continue
        
        # Multiple entries with same name
        # Separate into entries with metadata and without
        with_metadata = [e for e in entries if e.get('url') or e.get('version')]
        without_metadata = [e for e in entries if not (e.get('url') or e.get('version'))]
        
        # If we have entries with metadata, use only those (discard standalone)
        if with_metadata:
            # Deduplicate by name+url, keep the most complete entry (one with most fields)
            unique_metadata = {}
            for entry in with_metadata:
                # Key: name + url (not version, so versions are compared)
                key = (entry.get('name', '').lower().strip(), entry.get('url', ''))
                
                if key not in unique_metadata:
                    unique_metadata[key] = entry
                else:
                    # Keep the entry with more metadata
                    existing = unique_metadata[key]
                    existing_completeness = sum([bool(existing.get('url')), bool(existing.get('version'))])
                    new_completeness = sum([bool(entry.get('url')), bool(entry.get('version'))])
                    
                    if new_completeness > existing_completeness:
                        unique_metadata[key] = entry
            
            result.extend(unique_metadata.values())
        else:
            # All are standalone, keep just one
            result.append(entries[0])
    
    return result

def process_ground_truth_files():
    if not os.path.exists(GROUND_TRUTH_DIR):
        print(f"Error: Ground truth directory '{GROUND_TRUTH_DIR}' not found")
        return {}
    project_annotations = defaultdict(list)
    processed_files = 0
    for filename in os.listdir(GROUND_TRUTH_DIR):
        if not filename.endswith('.json'):
            continue
        filepath = os.path.join(GROUND_TRUTH_DIR, filename)
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            processed_files += 1
            for item in data:
                repo_field = item.get('data', {}).get('repo', '')
                project_name = extract_project_name(repo_field)
                for annotation in item.get('annotations', []):
                    if annotation.get('result') and not annotation.get('was_cancelled', False):
                        project_annotations[project_name].append(annotation)
        except Exception as e:
            print(f'Error processing {filename}: {e}')

    generated = 0
    for project_name, annotations in project_annotations.items():
        if not annotations:
            continue
        codemeta = create_codemeta_structure(project_name, annotations)
        output_path = os.path.join(OUTPUT_DIR, f'{project_name}_groundtruth_codemeta.json')
        try:
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(codemeta, f, indent=2, ensure_ascii=False)
            generated += 1
            print(f'Generated: {output_path}')
        except Exception as e:
            print(f'Error saving {output_path}: {e}')

    print(f'Processed {processed_files} files')
    print(f'Generated {generated} CodeMeta files in {OUTPUT_DIR}')
    return project_annotations

## Run Verarbeitung (optional)

In [5]:
# Starte die Verarbeitung — auskommentieren, wenn du sie manuell laufen lassen willst
project_annotations = process_ground_truth_files()
print('Fertig — überprüfe ./codemeta_groundtruth')

Generated: ../pipelines/evaluation/extrated_codemeta_files/Cesium-ncWMS_groundtruth_codemeta.json
Generated: ../pipelines/evaluation/extrated_codemeta_files/byteparsing_groundtruth_codemeta.json
Generated: ../pipelines/evaluation/extrated_codemeta_files/AutoPQ_groundtruth_codemeta.json
Generated: ../pipelines/evaluation/extrated_codemeta_files/OpenSim_Creator_groundtruth_codemeta.json
Generated: ../pipelines/evaluation/extrated_codemeta_files/APE_groundtruth_codemeta.json
Generated: ../pipelines/evaluation/extrated_codemeta_files/qc2_groundtruth_codemeta.json
Generated: ../pipelines/evaluation/extrated_codemeta_files/LUE_groundtruth_codemeta.json
Generated: ../pipelines/evaluation/extrated_codemeta_files/ReSurfEMG_groundtruth_codemeta.json
Generated: ../pipelines/evaluation/extrated_codemeta_files/iBridges-GUI_groundtruth_codemeta.json
Generated: ../pipelines/evaluation/extrated_codemeta_files/admtools_groundtruth_codemeta.json
Generated: ../pipelines/evaluation/extrated_codemeta_files